# FedGATSage – Kaggle Execution Notebook

**Graph-based Federated Learning for IoT Intrusion Detection**

Paper: Scientific Reports (2025) — https://doi.org/10.1038/s41598-025-25175-1

This notebook is fully self-contained. It embeds all source modules inline and runs end-to-end on Kaggle using either the **CIC-ToN-IoT** or **NF-ToN-IoT** dataset.

---
**Architecture overview:**
- Three specialised GAT client models: *Temporal*, *Content*, *Behavioral*
- Server-side **GraphSAGE** that aggregates community flow embeddings
- Louvain community detection for privacy-preserving local abstraction
- Federated aggregation via FedAvg with flow-embedding exchange

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'torch-geometric', '-q'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'python-louvain', 'networkx', '-q'])

print('Dependencies installed.')

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import os
import glob
import time
import logging
import warnings
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'NumPy    : {np.__version__}')
print(f'Pandas   : {pd.__version__}')

## GNN Models

Four neural-network classes (source: `src/gnn_models.py`):

| Class | Role |
|---|---|
| `TemporalGATDetector` | Detects DDoS / DoS / Scanning via timing patterns |
| `ContentGATDetector` | Detects Injection / XSS via payload patterns |
| `BehavioralGATDetector` | Detects Backdoor / Password / Ransomware via session patterns |
| `GlobalGraphSAGE` | Server-side aggregation of community flow embeddings |

In [ ]:
from torch_geometric.nn import GATConv, SAGEConv


# ── TemporalGATDetector ───────────────────────────────────────────────────────
class TemporalGATDetector(nn.Module):
    """GAT specialised for temporal attacks (DDoS, DoS, Scanning)."""

    def __init__(self, input_dim: int, hidden_dim: int = 256,
                 num_heads: int = 8, num_classes: int = 2, dropout: float = 0.2):
        super().__init__()
        self.gat1 = GATConv(input_dim, hidden_dim // num_heads,
                            heads=num_heads, concat=True, dropout=dropout)
        self.gat2 = GATConv(hidden_dim, hidden_dim,
                            heads=1, concat=False, dropout=dropout)
        self.gat3 = GATConv(hidden_dim, hidden_dim,
                            heads=1, concat=False, dropout=dropout)
        self.temporal_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.edge_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
        self.skip_conn1 = nn.Linear(input_dim, hidden_dim)
        self.skip_conn2 = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        res1 = self.skip_conn1(x)
        x = self.gat1(x, edge_index)
        x = self.norm1(x); x = F.elu(x); x = self.dropout(x); x = x + res1
        temporal_weights = self.temporal_attention(x)
        x = x * temporal_weights
        res2 = self.skip_conn2(x)
        x = self.gat2(x, edge_index)
        x = self.norm2(x); x = F.elu(x); x = self.dropout(x); x = x + res2
        x = self.gat3(x, edge_index)
        x = self.norm3(x); x = F.elu(x); x = self.dropout(x)
        edge_src, edge_dst = edge_index
        edge_features = torch.cat([x[edge_src], x[edge_dst]], dim=1)
        return x, self.edge_classifier(edge_features)


# ── ContentGATDetector ────────────────────────────────────────────────────────
class ContentGATDetector(nn.Module):
    """GAT specialised for content attacks (Injection, XSS)."""

    def __init__(self, input_dim: int, hidden_dim: int = 256,
                 num_heads: int = 8, num_classes: int = 2, dropout: float = 0.2):
        super().__init__()
        self.gat1 = GATConv(input_dim, hidden_dim // num_heads,
                            heads=num_heads, concat=True, dropout=dropout)
        self.gat2 = GATConv(hidden_dim, hidden_dim,
                            heads=2, concat=False, dropout=dropout)
        self.gat3 = GATConv(hidden_dim, hidden_dim,
                            heads=2, concat=False, dropout=dropout)
        self.content_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.Sigmoid()
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.edge_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 3),
            nn.BatchNorm1d(hidden_dim * 3),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, num_classes)
        )
        self.skip_conn1 = nn.Linear(input_dim, hidden_dim)
        self.skip_conn2 = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        res1 = self.skip_conn1(x)
        x = self.gat1(x, edge_index)
        x = self.norm1(x); x = F.elu(x); x = self.dropout(x); x = x + res1
        content_weights = self.content_attention(x)
        x = x * content_weights
        res2 = self.skip_conn2(x)
        x = self.gat2(x, edge_index)
        x = self.norm2(x); x = F.elu(x); x = self.dropout(x); x = x + res2
        x = self.gat3(x, edge_index)
        x = self.norm3(x); x = F.elu(x); x = self.dropout(x)
        edge_src, edge_dst = edge_index
        edge_features = torch.cat([x[edge_src], x[edge_dst]], dim=1)
        return x, self.edge_classifier(edge_features)


# ── BehavioralGATDetector ─────────────────────────────────────────────────────
class BehavioralGATDetector(nn.Module):
    """GAT specialised for behavioral attacks (Backdoor, Password, Ransomware)."""

    def __init__(self, input_dim: int, hidden_dim: int = 256,
                 num_heads: int = 8, num_classes: int = 2, dropout: float = 0.2):
        super().__init__()
        self.gat1 = GATConv(input_dim, hidden_dim // num_heads,
                            heads=num_heads, concat=True, dropout=dropout)
        self.gat2 = GATConv(hidden_dim, hidden_dim,
                            heads=1, concat=False, dropout=dropout)
        self.gat3 = GATConv(hidden_dim, hidden_dim,
                            heads=1, concat=False, dropout=dropout)
        self.session_encoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid()
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.edge_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
        self.skip_conn1 = nn.Linear(input_dim, hidden_dim)
        self.skip_conn2 = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        res1 = self.skip_conn1(x)
        x = self.gat1(x, edge_index)
        x = self.norm1(x); x = F.elu(x); x = self.dropout(x); x = x + res1
        session_features = self.session_encoder(x)
        x = x * session_features
        res2 = self.skip_conn2(x)
        x = self.gat2(x, edge_index)
        x = self.norm2(x); x = F.elu(x); x = self.dropout(x); x = x + res2
        x = self.gat3(x, edge_index)
        x = self.norm3(x); x = F.elu(x); x = self.dropout(x)
        edge_src, edge_dst = edge_index
        edge_features = torch.cat([x[edge_src], x[edge_dst]], dim=1)
        return x, self.edge_classifier(edge_features)


# ── GlobalGraphSAGE ───────────────────────────────────────────────────────────
class GlobalGraphSAGE(nn.Module):
    """Server-side GraphSAGE for aggregating community flow embeddings."""

    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3)
        )
        self.sage1 = SAGEConv(hidden_dim * 2, hidden_dim)
        self.sage2 = SAGEConv(hidden_dim, hidden_dim // 2)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim // 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = self.input_projection(x)
        x = self.sage1(x, edge_index)
        x = self.bn1(x); x = F.leaky_relu(x, 0.2); x = self.dropout(x)
        x = self.sage2(x, edge_index)
        x = self.bn2(x); x = F.leaky_relu(x, 0.2); x = self.dropout(x)
        return x, self.classifier(x)


print('GNN model classes defined.')

## Feature Engineering

Source: `src/feature_engineering.py`

- `FeatureEngineer` — adds base flow features plus detector-specific features (temporal / content / behavioral)
- `CentralityFeatureExtractor` — pass-through that logs pre-computed centrality columns if present

In [ ]:
class FeatureEngineer:
    """
    Handles extraction of specialised features for each GAT detector.
    Temporal / Content / Behavioral models each receive the features
    most relevant to their attack category.
    """

    def __init__(self, detector_type: str = 'temporal'):
        self.detector_type = detector_type
        self.created_features: List[str] = []

    def extract_features(self, df: pd.DataFrame) -> pd.DataFrame:
        result_df = df.copy()
        result_df = self._add_base_features(result_df)
        if self.detector_type == 'temporal':
            result_df = self._add_temporal_features(result_df)
        elif self.detector_type == 'content':
            result_df = self._add_content_features(result_df)
        elif self.detector_type == 'behavioral':
            result_df = self._add_behavioral_features(result_df)
        logger.info(
            f'Engineered {len(self.created_features)} new features '
            f'for {self.detector_type} detection'
        )
        return result_df

    def _add_base_features(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'Flow Duration' in df.columns and 'Tot Fwd Pkts' in df.columns:
            df['flow_rate'] = (
                (df['Tot Fwd Pkts'] + df['Tot Bwd Pkts'])
                / (df['Flow Duration'] / 1_000_000 + 1e-6)
            )
            self.created_features.append('flow_rate')
        if 'TotLen Fwd Pkts' in df.columns and 'Tot Fwd Pkts' in df.columns:
            df['avg_payload_fwd'] = df['TotLen Fwd Pkts'] / (df['Tot Fwd Pkts'] + 1e-6)
            df['avg_payload_bwd'] = df['TotLen Bwd Pkts'] / (df['Tot Bwd Pkts'] + 1e-6)
            self.created_features.extend(['avg_payload_fwd', 'avg_payload_bwd'])
        if 'Protocol' in df.columns:
            df['protocol_encoded'] = pd.Categorical(df['Protocol'].astype(str)).codes
            self.created_features.append('protocol_encoded')
        return df

    def _add_temporal_features(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'Flow IAT Mean' in df.columns and 'Flow IAT Std' in df.columns:
            df['iat_variance'] = df['Flow IAT Std'] / (df['Flow IAT Mean'] + 1e-6)
            self.created_features.append('iat_variance')
        if 'Flow Pkts/s' in df.columns:
            mean_pps = df['Flow Pkts/s'].mean()
            df['burst_ratio'] = df['Flow Pkts/s'] / (mean_pps + 1e-6)
            df['is_burst'] = (df['burst_ratio'] > 2.0).astype(int)
            self.created_features.extend(['burst_ratio', 'is_burst'])
        flag_cols = ['SYN Flag Cnt', 'RST Flag Cnt', 'ACK Flag Cnt']
        if all(c in df.columns for c in flag_cols):
            df['syn_rst_ratio'] = df['SYN Flag Cnt'] / (df['RST Flag Cnt'] + 1e-6)
            df['unusual_flags'] = (
                (df['SYN Flag Cnt'] > 0) & (df['RST Flag Cnt'] > 0)
            ).astype(int)
            self.created_features.extend(['syn_rst_ratio', 'unusual_flags'])
        return df

    def _add_content_features(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'Dst Port' in df.columns:
            df['is_web_port'] = df['Dst Port'].isin([80, 443, 8080, 8443]).astype(int)
            df['is_db_port'] = df['Dst Port'].isin([1433, 1521, 3306, 5432]).astype(int)
            self.created_features.extend(['is_web_port', 'is_db_port'])
        if 'TotLen Fwd Pkts' in df.columns:
            mean_p = df['TotLen Fwd Pkts'].mean()
            std_p = df['TotLen Fwd Pkts'].std()
            df['unusual_payload'] = (df['TotLen Fwd Pkts'] > mean_p + 2 * std_p).astype(int)
            df['payload_ratio'] = df['TotLen Fwd Pkts'] / (
                df['TotLen Bwd Pkts'] + 1e-6
            )
            self.created_features.extend(['unusual_payload', 'payload_ratio'])
        return df

    def _add_behavioral_features(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'Src Port' in df.columns and 'Dst Port' in df.columns:
            df['is_ephemeral_src'] = (df['Src Port'] > 1024).astype(int)
            df['targets_system_port'] = (df['Dst Port'] < 1024).astype(int)
            df['port_spread'] = abs(df['Src Port'] - df['Dst Port'])
            self.created_features.extend(
                ['is_ephemeral_src', 'targets_system_port', 'port_spread']
            )
        if 'Flow Duration' in df.columns:
            med = df['Flow Duration'].median()
            df['is_short_session'] = (df['Flow Duration'] < med / 10).astype(int)
            df['is_long_session'] = (df['Flow Duration'] > med * 10).astype(int)
            self.created_features.extend(['is_short_session', 'is_long_session'])
        if 'TotLen Fwd Pkts' in df.columns:
            df['is_low_volume'] = (
                (df['TotLen Fwd Pkts'] < 100) & (df['Tot Fwd Pkts'] < 5)
            ).astype(int)
            self.created_features.append('is_low_volume')
        return df


class CentralityFeatureExtractor:
    """Pass-through extractor that logs pre-computed centrality columns."""

    def __init__(self):
        self.centrality_cache: Dict = {}

    def extract_centrality_features(self, df: pd.DataFrame) -> pd.DataFrame:
        centrality_cols = [
            c for c in df.columns
            if any(m in c.lower() for m in [
                'betweenness', 'pagerank', 'degree', 'closeness',
                'eigenvector', 'k_core', 'k_truss', 'modularity'
            ])
        ]
        if centrality_cols:
            logger.info(
                f'Found {len(centrality_cols)} centrality features: {centrality_cols}'
            )
        else:
            logger.warning('No centrality features found in dataset')
        return df


print('Feature engineering classes defined.')

## Community Detection

Source: `src/community_detection.py`

`CommunityAwareProcessor` implements **Algorithm 1** from the paper:
1. Build flow graph from raw dataframe
2. Detect communities with Louvain (falls back to greedy modularity)
3. Compute **modularity vitality** per node
4. Attach community features (`src_community`, `dst_community`, `is_inter_community`, etc.) to the dataframe

In [ ]:
import networkx as nx


class CommunityAwareProcessor:
    """
    Bridges the Community Abstraction concept (Algorithm 1, paper) with the
    practical implementation.

    Steps:
      1. Build graph from flows
      2. Detect communities via Louvain
      3. Compute modularity vitality per node
      4. Attach community-aware columns to dataframe
    """

    def __init__(self):
        self.communities = None
        self.modularity_vitality = None

    # ------------------------------------------------------------------
    def detect_communities_louvain(self, graph: nx.Graph) -> Dict:
        """Community detection – Algorithm 1, Step 2."""
        try:
            import community.community_louvain as community_louvain
            partition = community_louvain.best_partition(graph)
            self.communities = partition
            logger.info(f'Detected {len(set(partition.values()))} communities (Louvain)')
            return partition
        except ImportError:
            logger.warning('python-louvain not available; using NetworkX greedy fallback')
            return self._networkx_community_detection(graph)

    def _networkx_community_detection(self, graph: nx.Graph) -> Dict:
        from networkx.algorithms.community import greedy_modularity_communities
        communities = list(greedy_modularity_communities(graph))
        partition = {}
        for idx, comm in enumerate(communities):
            for node in comm:
                partition[node] = idx
        self.communities = partition
        logger.info(f'Detected {len(communities)} communities (greedy modularity)')
        return partition

    def _partition_to_community_sets(self, communities: Dict) -> List[set]:
        """Convert {node: comm_id} -> [{nodes_in_comm}, ...] for nx.modularity."""
        comm_to_nodes: Dict = {}
        for node, comm_id in communities.items():
            comm_to_nodes.setdefault(comm_id, set()).add(node)
        return list(comm_to_nodes.values())

    # ------------------------------------------------------------------
    def compute_modularity_vitality(
        self, graph: nx.Graph, communities: Dict
    ) -> Dict:
        """Modularity vitality per node – Algorithm 1, Step 2."""
        modularity_vitality: Dict = {}
        community_sets = self._partition_to_community_sets(communities)
        try:
            base_modularity = nx.community.modularity(graph, community_sets)
        except Exception:
            base_modularity = 0.0

        for node in graph.nodes():
            temp_graph = graph.copy()
            temp_graph.remove_node(node)
            temp_communities = {k: v for k, v in communities.items() if k != node}
            if len(temp_communities) > 0:
                try:
                    temp_sets = self._partition_to_community_sets(temp_communities)
                    new_mod = nx.community.modularity(temp_graph, temp_sets)
                    modularity_vitality[node] = base_modularity - new_mod
                except Exception:
                    modularity_vitality[node] = 0.0
            else:
                modularity_vitality[node] = 0.0

        self.modularity_vitality = modularity_vitality
        return modularity_vitality

    # ------------------------------------------------------------------
    def create_community_enhanced_features(
        self, df: pd.DataFrame, ip_to_idx: Dict
    ) -> pd.DataFrame:
        """Add community-aware columns to dataframe."""
        G = nx.Graph()
        for _, row in df.iterrows():
            src, dst = row['Src IP'], row['Dst IP']
            if G.has_edge(src, dst):
                G[src][dst]['weight'] += 1
            else:
                G.add_edge(src, dst, weight=1)

        communities = self.detect_communities_louvain(G)
        mod_vitality = self.compute_modularity_vitality(G, communities)

        df_enhanced = df.copy()
        df_enhanced['src_community'] = df_enhanced['Src IP'].map(communities)
        df_enhanced['dst_community'] = df_enhanced['Dst IP'].map(communities)
        df_enhanced['src_mod_vitality'] = df_enhanced['Src IP'].map(mod_vitality)
        df_enhanced['dst_mod_vitality'] = df_enhanced['Dst IP'].map(mod_vitality)
        df_enhanced['is_inter_community'] = (
            df_enhanced['src_community'] != df_enhanced['dst_community']
        ).astype(int)

        logger.info(f'Enhanced {len(df_enhanced)} flows with community features')
        return df_enhanced

    # ------------------------------------------------------------------
    def aggregate_to_community_embeddings(
        self, node_embeddings: np.ndarray, communities: Dict
    ) -> Dict:
        """Aggregate node embeddings to community level – Algorithm 1, Step 5."""
        community_embeddings: Dict = {}
        for comm_id in set(communities.values()):
            nodes = [n for n, c in communities.items() if c == comm_id]
            if nodes:
                community_embeddings[comm_id] = np.mean(
                    [node_embeddings[n] for n in nodes], axis=0
                )
        return community_embeddings


print('Community detection class defined.')

## Federated Learning Core

Source: `src/federated_learning.py`

Three classes:

| Class | Responsibility |
|---|---|
| `FlowEmbeddingGenerator` | Converts GAT node embeddings → privacy-safe flow embeddings (community abstractions) |
| `DataLoader` | Loads client CSV, engineers features, builds graph tensors |
| `FedGATSageSystem` | Orchestrates federated rounds: collect → aggregate → redistribute |

> **Notebook fix (requirement 6):** The original `_aggregate_updates` used `torch.combinations` to build a fully-connected graph, which is O(n²) in memory. This notebook replaces it with a **sparse k-NN graph** (k=10, max 512 nodes) built from cosine similarity, eliminating OOM errors on Kaggle.

In [ ]:
# ── FlowEmbeddingGenerator ────────────────────────────────────────────────────
class FlowEmbeddingGenerator:
    """Generates flow embeddings as community abstractions (Algorithm 1, Step 4)."""

    def __init__(self, detector_type: str = 'temporal'):
        self.detector_type = detector_type

    def generate_embeddings(
        self, model: nn.Module, data: Dict[str, Any]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Generate flow embeddings from GAT node embeddings."""
        model.eval()
        with torch.no_grad():
            device = next(model.parameters()).device
            x = data['features'].to(device)
            edge_index = data['edge_index'].to(device)
            edge_labels = data['edge_labels'].to(device)

            logger.info(
                f'Generating embeddings: {x.shape[0]} nodes, '
                f'{edge_index.shape[1]} edges'
            )
            try:
                node_embeddings, _ = model(x, edge_index)
            except Exception as e:
                logger.error(f'GAT forward pass error: {e}')
                return torch.empty(0), torch.empty(0)

            flow_embeddings = []
            flow_labels = []
            unique_labels = torch.unique(edge_labels)
            max_per_class = min(250, max(1, len(edge_labels) // len(unique_labels)))

            for label in unique_labels:
                mask = edge_labels == label
                label_indices = mask.nonzero(as_tuple=True)[0]
                if len(label_indices) > max_per_class:
                    perm = torch.randperm(len(label_indices))[:max_per_class]
                    selected_indices = label_indices[perm]
                else:
                    selected_indices = label_indices
                for idx in selected_indices:
                    src_idx = edge_index[0, idx]
                    dst_idx = edge_index[1, idx]
                    flow_emb = self._create_flow_embedding(
                        node_embeddings[src_idx],
                        node_embeddings[dst_idx],
                        data, idx
                    )
                    flow_embeddings.append(flow_emb.unsqueeze(0))
                    flow_labels.append(label)

            if flow_embeddings:
                flow_embeddings = torch.cat(flow_embeddings, dim=0)
                flow_labels = torch.stack(flow_labels)
                logger.info(f'Generated {len(flow_embeddings)} flow embeddings')
                return flow_embeddings, flow_labels
            else:
                logger.warning('No flow embeddings generated')
                return torch.empty(0), torch.empty(0)

    def _create_flow_embedding(
        self,
        src_emb: torch.Tensor,
        dst_emb: torch.Tensor,
        data: Dict[str, Any],
        idx: int
    ) -> torch.Tensor:
        """Create flow embedding representing community relationship."""
        parts = [
            src_emb,
            dst_emb,
            src_emb * dst_emb,          # element-wise product
            torch.abs(src_emb - dst_emb) # absolute difference
        ]
        if 'traffic_features' in data and data['traffic_features'] is not None:
            parts.append(data['traffic_features'][idx].to(src_emb.device))
        return torch.cat(parts)


print('FlowEmbeddingGenerator defined.')

In [ ]:
# ── DataLoader ────────────────────────────────────────────────────────────────
class DataLoader:
    """
    Loads and processes client CSV files into graph tensors for GNN training.

    The row-iteration in _process_to_graph is replaced by vectorised
    pandas operations to keep wall-clock time reasonable on Kaggle.
    """

    def __init__(self, data_dir: str, detector_type: str = 'temporal'):
        self.data_dir = data_dir
        self.detector_type = detector_type
        self.feature_engineer = FeatureEngineer(detector_type)
        self.centrality_extractor = CentralityFeatureExtractor()
        self.community_processor = CommunityAwareProcessor()
        self.label_mapper: Optional[Dict] = None

    # ------------------------------------------------------------------
    def load_client_data(self, client_id: int) -> Optional[Dict[str, Any]]:
        """Load and process one client's CSV into graph tensors."""
        client_path = os.path.join(self.data_dir, f'client_{client_id}.csv')
        if not os.path.exists(client_path):
            logger.error(f'Client file not found: {client_path}')
            return None
        try:
            df = pd.read_csv(client_path)
            logger.info(f'Loaded {len(df)} records for client {client_id}')
            if self.label_mapper is None:
                self._create_label_mapper(df)
            df = self.feature_engineer.extract_features(df)
            df = self.centrality_extractor.extract_centrality_features(df)
            df = self.community_processor.create_community_enhanced_features(df, {})
            return self._process_to_graph(df)
        except Exception as e:
            logger.error(f'Error loading client {client_id}: {e}')
            return None

    def _create_label_mapper(self, df: pd.DataFrame):
        unique_attacks = sorted(df['Attack'].unique())
        self.label_mapper = {a: i for i, a in enumerate(unique_attacks)}
        logger.info(f'Label mapper: {self.label_mapper}')

    def _process_to_graph(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Convert DataFrame to graph-format tensors (vectorised, no row loops)."""
        unique_ips = pd.concat([df['Src IP'], df['Dst IP']]).unique()
        ip_to_idx = {ip: i for i, ip in enumerate(unique_ips)}

        feature_cols = [
            c for c in df.columns
            if any(m in c.lower() for m in [
                'betweenness', 'pagerank', 'degree', 'closeness', 'eigenvector',
                'k_core', 'k_truss', 'modularity', 'flow_rate', 'avg_payload'
            ])
        ]
        if not feature_cols:
            feature_cols = ['flow_rate', 'avg_payload_fwd', 'protocol_encoded']
            for col in feature_cols:
                if col not in df.columns:
                    df[col] = 0.0

        # Vectorised node-feature aggregation
        src_feats = df[['Src IP'] + feature_cols].rename(columns={'Src IP': 'ip'})
        dst_feats = df[['Dst IP'] + feature_cols].rename(columns={'Dst IP': 'ip'})
        all_feats = pd.concat([src_feats, dst_feats])
        node_feat_df = all_feats.groupby('ip')[feature_cols].mean().reindex(unique_ips)
        node_feat_df = node_feat_df.fillna(0.0)
        features = torch.tensor(node_feat_df.values, dtype=torch.float32)

        # Vectorised edge construction
        valid_mask = df['Src IP'].isin(ip_to_idx) & df['Dst IP'].isin(ip_to_idx)
        valid_df = df[valid_mask]
        src_indices = valid_df['Src IP'].map(ip_to_idx).values
        dst_indices = valid_df['Dst IP'].map(ip_to_idx).values
        edge_label_vals = valid_df['Attack'].map(self.label_mapper).values

        edge_index = torch.tensor(
            np.stack([src_indices, dst_indices], axis=0), dtype=torch.long
        )
        edge_labels = torch.tensor(edge_label_vals, dtype=torch.long)

        return {
            'features': features,
            'edge_index': edge_index,
            'edge_labels': edge_labels,
            'ip_to_idx': ip_to_idx,
            'df': df
        }


print('DataLoader defined.')

In [ ]:
# ── FedGATSageSystem ──────────────────────────────────────────────────────────
# NOTE: _aggregate_updates uses a sparse k-NN graph (k=10, max 512 nodes)
# instead of torch.combinations, preventing O(n²) OOM on large batches.

class FedGATSageSystem:
    """Main FedGATSage federated learning orchestrator."""

    def __init__(
        self,
        data_dir: str,
        num_clients: int = 5,
        detector_types: List[str] = None,
        device: str = 'cpu'
    ):
        if detector_types is None:
            detector_types = ['temporal', 'content', 'behavioral']
        self.data_dir = data_dir
        self.num_clients = num_clients
        self.detector_types = detector_types
        self.device = device

        self.client_models: Dict[str, Dict] = {}
        self.data_loaders: Dict[str, DataLoader] = {}
        self.flow_generators: Dict[str, FlowEmbeddingGenerator] = {}

        for dt in detector_types:
            detector_dir = os.path.join(data_dir, f'{dt}_detector')
            self.data_loaders[dt] = DataLoader(detector_dir, dt)
            self.flow_generators[dt] = FlowEmbeddingGenerator(dt)
            self.client_models[dt] = {}

        self.global_model: Optional[GlobalGraphSAGE] = None
        self.results: Dict[str, List] = {'training_losses': [], 'round_times': []}
        logger.info(
            f'FedGATSage initialised — {num_clients} clients, '
            f'{len(detector_types)} detector types, device={device}'
        )

    # ------------------------------------------------------------------
    def initialize_models(
        self, input_dim: int = 64, hidden_dim: int = 256, num_classes: int = 8
    ):
        """Initialise all client GAT models and the server GraphSAGE."""
        for dt in self.detector_types:
            for cid in range(self.num_clients):
                if dt == 'temporal':
                    m = TemporalGATDetector(input_dim, hidden_dim,
                                           num_classes=num_classes)
                elif dt == 'content':
                    m = ContentGATDetector(input_dim, hidden_dim,
                                          num_classes=num_classes)
                else:
                    m = BehavioralGATDetector(input_dim, hidden_dim,
                                             num_classes=num_classes)
                self.client_models[dt][cid] = m.to(self.device)

        # Determine flow-embedding dimension from a sample forward pass
        sample_data = self.data_loaders[self.detector_types[0]].load_client_data(1)
        flow_embedding_dim = hidden_dim * 4  # default fallback
        if sample_data is not None:
            sample_model = self.client_models[self.detector_types[0]][0]
            flow_gen = self.flow_generators[self.detector_types[0]]
            with torch.no_grad():
                sample_embs, _ = flow_gen.generate_embeddings(sample_model, sample_data)
                if len(sample_embs) > 0:
                    flow_embedding_dim = sample_embs.shape[1]

        self.global_model = GlobalGraphSAGE(
            input_dim=flow_embedding_dim,
            hidden_dim=hidden_dim,
            num_classes=num_classes
        ).to(self.device)
        logger.info(f'Models initialised. Flow embedding dim: {flow_embedding_dim}')

    # ------------------------------------------------------------------
    def train_federated(self, num_rounds: int = 20) -> Dict[str, Any]:
        """Main federated training loop."""
        logger.info(f'Starting federated training for {num_rounds} rounds')
        for rnd in range(num_rounds):
            t0 = time.time()
            logger.info(f'--- Round {rnd + 1}/{num_rounds} ---')
            all_updates: List[Dict] = []
            for dt in self.detector_types:
                all_updates.extend(self._collect_client_updates(dt))
            global_loss = self._aggregate_updates(all_updates)
            self._redistribute_models()
            elapsed = time.time() - t0
            self.results['training_losses'].append(global_loss)
            self.results['round_times'].append(elapsed)
            logger.info(
                f'Round {rnd + 1} done in {elapsed:.1f}s  loss={global_loss:.4f}'
            )
        logger.info('Federated training complete')
        return self.results

    # ------------------------------------------------------------------
    def _collect_client_updates(self, detector_type: str) -> List[Dict]:
        updates = []
        for cid in range(self.num_clients):
            client_data = self.data_loaders[detector_type].load_client_data(cid + 1)
            if client_data is None:
                continue
            model = self.client_models[detector_type][cid]
            metrics = self._train_client_model(model, client_data)
            flow_gen = self.flow_generators[detector_type]
            flow_embs, flow_labels = flow_gen.generate_embeddings(model, client_data)
            if len(flow_embs) > 0:
                updates.append({
                    'client_id': cid,
                    'detector_type': detector_type,
                    'flow_embeddings': flow_embs,
                    'flow_labels': flow_labels,
                    'model_state': model.state_dict(),
                    'metrics': metrics
                })
        return updates

    def _train_client_model(
        self, model: nn.Module, data: Dict[str, Any]
    ) -> Dict[str, float]:
        model.train()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        x = data['features'].to(self.device)
        edge_index = data['edge_index'].to(self.device)
        edge_labels = data['edge_labels'].to(self.device)
        loss_val = 0.0
        for _ in range(5):
            optimizer.zero_grad()
            _, preds = model(x, edge_index)
            loss = criterion(preds, edge_labels)
            loss.backward()
            optimizer.step()
            loss_val = loss.item()
        return {'loss': loss_val}

    # ------------------------------------------------------------------
    def _build_knn_edge_index(
        self, embeddings: torch.Tensor, k: int = 10, max_nodes: int = 512
    ) -> torch.Tensor:
        """
        Build a sparse k-NN graph from cosine similarity.
        At most `max_nodes` nodes are used; excess nodes are randomly sampled.
        This replaces the original torch.combinations call which was O(n^2).
        """
        n = embeddings.shape[0]
        if n > max_nodes:
            idx = torch.randperm(n)[:max_nodes]
            embeddings = embeddings[idx]
            n = max_nodes

        # Normalise for cosine similarity
        norms = embeddings.norm(dim=1, keepdim=True).clamp(min=1e-8)
        normed = embeddings / norms
        sim = normed @ normed.t()                     # [n, n]

        k_actual = min(k + 1, n)                      # +1 because self-loop excluded
        _, top_idx = sim.topk(k_actual, dim=1)        # [n, k_actual]

        src_list, dst_list = [], []
        for i in range(n):
            for j in top_idx[i]:
                j_int = j.item()
                if j_int != i:
                    src_list.append(i)
                    dst_list.append(j_int)

        if not src_list:                              # degenerate: single node
            return torch.zeros((2, 0), dtype=torch.long, device=embeddings.device)

        return torch.tensor(
            [src_list, dst_list], dtype=torch.long, device=embeddings.device
        )

    def _aggregate_updates(self, client_updates: List[Dict]) -> float:
        """Aggregate using global GraphSAGE on a sparse k-NN flow graph."""
        if not client_updates:
            return 0.0
        all_embs = torch.cat(
            [u['flow_embeddings'].to(self.device) for u in client_updates], dim=0
        )
        all_labels = torch.cat(
            [u['flow_labels'].to(self.device) for u in client_updates], dim=0
        )

        # Sparse k-NN graph (k=10, max 512 nodes) — no OOM risk
        edge_index = self._build_knn_edge_index(all_embs, k=10, max_nodes=512)

        # Truncate embeddings/labels to match sampled node count
        n = edge_index.max().item() + 1 if edge_index.numel() > 0 else all_embs.shape[0]
        global_x = all_embs[:n]
        global_y = all_labels[:n]

        self.global_model.train()
        optimizer = torch.optim.Adam(self.global_model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        optimizer.zero_grad()
        _, preds = self.global_model(global_x, edge_index)
        loss = criterion(preds, global_y)
        loss.backward()
        optimizer.step()
        return loss.item()

    def _redistribute_models(self):
        """FedAvg: average client models within each detector type."""
        for dt in self.detector_types:
            states = [
                self.client_models[dt][cid].state_dict()
                for cid in self.client_models[dt]
            ]
            if not states:
                continue
            avg_state = {}
            for key in states[0]:
                stacked = torch.stack([s[key] for s in states])
                if stacked.is_floating_point():
                    avg_state[key] = stacked.mean(0)
                else:
                    avg_state[key] = stacked.float().mean(0).to(stacked.dtype)
            for cid in self.client_models[dt]:
                self.client_models[dt][cid].load_state_dict(avg_state)


print('FedGATSageSystem defined.')

## Dataset Loading

Two datasets are supported:

| Dataset | Notes |
|---|---|
| **CIC-ToN-IoT** | Already has CIC-style column names; no renaming needed |
| **NF-ToN-IoT** | Uses IPFIX/NetFlow column names (e.g. `IPV4_SRC_ADDR`); must be fixed before use |

The cell below auto-detects whichever CSV is present under `/kaggle/input/`.

To add either dataset on Kaggle:
1. Go to **+ Add Data** in the right sidebar
2. Search for **"ToN-IoT"**
3. Attach the dataset and re-run

In [ ]:
def find_dataset(name_patterns: List[str]) -> Optional[str]:
    """Search /kaggle/input recursively for a file matching any of the patterns."""
    for pattern in name_patterns:
        matches = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
        if matches:
            return matches[0]
    return None


cic_path = find_dataset(['CIC-ToN-IoT.csv', 'CIC_ToN_IoT.csv', 'cictoniot.csv',
                         'CIC-ToN-IoT-*.csv', 'cic_ton_iot.csv'])
nf_path  = find_dataset(['NF-ToN-IoT.csv',  'NF_ToN_IoT.csv',  'nftoniot.csv',
                         'NF-ToN-IoT-*.csv',  'nf_ton_iot.csv'])

print(f'CIC-ToN-IoT : {cic_path}')
print(f'NF-ToN-IoT  : {nf_path}')

In [ ]:
# ── NF-ToN-IoT column-fix function ───────────────────────────────────────────
# Inline version of fix-NF-TON-IoT-dataset.py
# Renames NetFlow/IPFIX columns to CIC-style names and derives missing columns.

NF_RENAME_MAP = {
    'IPV4_SRC_ADDR':              'Src IP',
    'IPV4_DST_ADDR':              'Dst IP',
    'L4_SRC_PORT':                'Src Port',
    'L4_DST_PORT':                'Dst Port',
    'PROTOCOL':                   'Protocol',
    'FLOW_DURATION_MILLISECONDS': 'Flow Duration',
    'IN_PKTS':                    'Tot Fwd Pkts',
    'OUT_PKTS':                   'Tot Bwd Pkts',
    'IN_BYTES':                   'TotLen Fwd Pkts',
    'OUT_BYTES':                  'TotLen Bwd Pkts',
}

_SYN_BIT = 0x02
_RST_BIT = 0x04
_ACK_BIT = 0x10


def _nf_derive_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Derive columns required by FedGATSage from the NF schema (pre-rename)."""
    duration_s = df['FLOW_DURATION_MILLISECONDS'] / 1000.0
    total_pkts = df['IN_PKTS'] + df['OUT_PKTS']
    df['Flow Pkts/s'] = total_pkts / (duration_s + 1e-9)
    df['Flow IAT Mean'] = 0.0
    df['Flow IAT Std'] = 0.0
    if 'TCP_FLAGS' in df.columns:
        flags = df['TCP_FLAGS'].fillna(0).astype(int)
        df['SYN Flag Cnt'] = ((flags & _SYN_BIT) > 0).astype(int)
        df['RST Flag Cnt'] = ((flags & _RST_BIT) > 0).astype(int)
        df['ACK Flag Cnt'] = ((flags & _ACK_BIT) > 0).astype(int)
    else:
        df['SYN Flag Cnt'] = 0
        df['RST Flag Cnt'] = 0
        df['ACK Flag Cnt'] = 0
    return df


def fix_nf_toniot_dataset(
    input_path: str,
    output_path: str = '/kaggle/working/NF-ToN-IoT-fixed.csv',
    chunk_size: int = 100_000
) -> str:
    """
    Read NF-ToN-IoT CSV in chunks, derive missing columns, rename to CIC style,
    and write to output_path.

    Returns the output path on success.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    logger.info(f'Fixing NF-ToN-IoT: {input_path} -> {output_path}')
    total_rows = 0
    first_chunk = True
    for i, chunk in enumerate(pd.read_csv(input_path, chunksize=chunk_size,
                                          low_memory=False)):
        chunk = _nf_derive_columns(chunk)
        chunk.rename(columns=NF_RENAME_MAP, inplace=True)
        mode = 'w' if first_chunk else 'a'
        chunk.to_csv(output_path, index=False, mode=mode, header=first_chunk)
        total_rows += len(chunk)
        first_chunk = False
        logger.info(f'  chunk {i + 1}: {total_rows:,} rows written')
    logger.info(f'NF-ToN-IoT fix complete. Total rows: {total_rows:,}')
    return output_path


print('NF-ToN-IoT fix function defined.')

## Configuration

Edit `DATASET` to `'cic'` or `'nf'` depending on which dataset you have attached.
All other parameters can be left at their defaults for a standard Kaggle run.

In [ ]:
# ── Experiment configuration ──────────────────────────────────────────────────
DATASET          = 'cic'    # 'cic' or 'nf'
NUM_CLIENTS      = 5
NUM_ROUNDS       = 15
HIDDEN_DIM       = 256
NUM_HEADS        = 8
DROPOUT          = 0.2
DETECTOR_TYPES   = ['temporal', 'content', 'behavioral']
MAX_SAMPLE_ROWS  = 200_000   # stratified sample cap before preprocessing
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_OUTPUT_DIR  = '/kaggle/working/data'
RESULTS_DIR      = '/kaggle/working/results'
SEED             = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(DATA_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Device         : {DEVICE}')
print(f'Dataset        : {DATASET}')
print(f'Clients        : {NUM_CLIENTS}')
print(f'Rounds         : {NUM_ROUNDS}')
print(f'Max sample rows: {MAX_SAMPLE_ROWS:,}')

In [ ]:
# ── Data preprocessing helpers ────────────────────────────────────────────────

def stratified_sample(df: pd.DataFrame, max_rows: int, seed: int = 42) -> pd.DataFrame:
    """
    Return at most `max_rows` rows sampled proportionally by Attack label.
    Preserves the class distribution of the original dataset.
    """
    if len(df) <= max_rows:
        return df
    frac = max_rows / len(df)
    sampled = (
        df.groupby('Attack', group_keys=False)
          .apply(lambda g: g.sample(frac=frac, random_state=seed))
    )
    # Ensure we do not exceed max_rows due to rounding
    return sampled.sample(n=min(max_rows, len(sampled)),
                          random_state=seed).reset_index(drop=True)


def save_split_data(
    df: pd.DataFrame,
    output_dir: str,
    num_clients: int,
    seed: int = 42
):
    """
    Split df into train/test (80/20) and distribute training data
    among `num_clients` CSV files.  Mirrors preprocess_data.py logic.
    """
    os.makedirs(output_dir, exist_ok=True)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=seed)

    test_path = os.path.join(output_dir, 'test.csv')
    test_df.to_csv(test_path, index=False)
    logger.info(f'Saved test set  : {test_path}  ({len(test_df):,} rows)')

    client_dfs = np.array_split(train_df, num_clients)
    for i, cdf in enumerate(client_dfs):
        client_path = os.path.join(output_dir, f'client_{i + 1}.csv')
        cdf.to_csv(client_path, index=False)
        logger.info(f'Saved client {i + 1}: {client_path}  ({len(cdf):,} rows)')


print('Preprocessing helpers defined.')

In [ ]:
# ── Run preprocessing for the selected dataset ────────────────────────────────

# Resolve input CSV path
if DATASET == 'cic':
    raw_csv = cic_path
    dataset_label = 'CIC-ToN-IoT'
elif DATASET == 'nf':
    raw_csv = nf_path
    dataset_label = 'NF-ToN-IoT'
else:
    raise ValueError(f'Unknown DATASET value: {DATASET!r}. Must be "cic" or "nf".')

# If no real dataset found, generate a synthetic one for demonstration
if raw_csv is None:
    logger.warning(
        f'No {dataset_label} CSV found under /kaggle/input/. '
        f'Generating a synthetic dataset for demonstration.'
    )
    _n = 5_000
    _rng = np.random.default_rng(SEED)
    _attack_choices = ['Benign', 'DoS', 'DDoS', 'PortScan', 'WebAttack',
                       'Backdoor', 'Injection', 'Password']
    _attack_probs   = [0.50, 0.10, 0.10, 0.08, 0.07, 0.06, 0.05, 0.04]
    synthetic = pd.DataFrame({
        'Src IP':          [f'192.168.{_rng.integers(1,5)}.{_rng.integers(1,255)}'
                            for _ in range(_n)],
        'Dst IP':          [f'10.0.{_rng.integers(0,4)}.{_rng.integers(1,255)}'
                            for _ in range(_n)],
        'Src Port':        _rng.integers(1024, 65535, _n),
        'Dst Port':        _rng.integers(1, 1024, _n),
        'Protocol':        _rng.choice(['TCP', 'UDP'], _n),
        'Flow Duration':   _rng.integers(100, 100_000, _n),
        'Tot Fwd Pkts':    _rng.integers(1, 100, _n),
        'Tot Bwd Pkts':    _rng.integers(1, 100, _n),
        'TotLen Fwd Pkts': _rng.integers(64, 15_000, _n),
        'TotLen Bwd Pkts': _rng.integers(64, 15_000, _n),
        'Flow IAT Mean':   _rng.uniform(0.1, 100.0, _n),
        'Flow IAT Std':    _rng.uniform(0.0, 10.0, _n),
        'Flow Pkts/s':     _rng.uniform(0.1, 1000.0, _n),
        'SYN Flag Cnt':    _rng.integers(0, 2, _n),
        'RST Flag Cnt':    _rng.integers(0, 2, _n),
        'ACK Flag Cnt':    _rng.integers(0, 2, _n),
        'Attack':          _rng.choice(_attack_choices, _n, p=_attack_probs),
    })
    # Add dummy centrality columns
    for _metric in ['betweenness', 'pagerank', 'degree', 'modularity']:
        synthetic[f'src_{_metric}'] = _rng.uniform(0, 1, _n)
        synthetic[f'dst_{_metric}'] = _rng.uniform(0, 1, _n)
    synthetic_path = '/kaggle/working/synthetic_dataset.csv'
    synthetic.to_csv(synthetic_path, index=False)
    raw_csv = synthetic_path
    logger.info(f'Synthetic dataset saved to {synthetic_path} ({_n:,} rows)')

# Apply NF-ToN-IoT column fix if needed
if DATASET == 'nf' and raw_csv != '/kaggle/working/NF-ToN-IoT-fixed.csv':
    logger.info('Applying NF-ToN-IoT column fix ...')
    raw_csv = fix_nf_toniot_dataset(
        raw_csv, '/kaggle/working/NF-ToN-IoT-fixed.csv'
    )

# Load, sample, and split
logger.info(f'Loading {raw_csv} ...')
df_raw = pd.read_csv(raw_csv, low_memory=False)
logger.info(f'Loaded {len(df_raw):,} rows, {df_raw.shape[1]} columns')

# Drop rows with missing Attack label
df_raw = df_raw.dropna(subset=['Attack'])
logger.info(f'After dropping NaN Attack: {len(df_raw):,} rows')

# Stratified subsample (max 200k rows to keep Kaggle run time manageable)
df_sampled = stratified_sample(df_raw, MAX_SAMPLE_ROWS, seed=SEED)
logger.info(
    f'Stratified sample: {len(df_sampled):,} rows  '
    f'(classes: {df_sampled["Attack"].nunique()})'
)
print(df_sampled['Attack'].value_counts())

# Write detector-specific client splits
for detector in DETECTOR_TYPES:
    detector_dir = os.path.join(DATA_OUTPUT_DIR, f'{detector}_detector')
    logger.info(f'Splitting data for {detector} detector -> {detector_dir}')
    save_split_data(df_sampled, detector_dir, NUM_CLIENTS, seed=SEED)

logger.info('Preprocessing complete.')

## Training

Initialise all client models and the server GraphSAGE, then run the federated training loop.

Each round:
1. Each client trains its local GAT for 5 epochs
2. Clients send **flow embeddings** (privacy-safe community abstractions) to the server
3. Server trains GlobalGraphSAGE on a k-NN graph built from those embeddings
4. Updated client models are averaged (FedAvg) and redistributed

In [ ]:
# ── Determine input_dim from actual data ──────────────────────────────────────
_probe_path = os.path.join(DATA_OUTPUT_DIR, 'temporal_detector', 'client_1.csv')
_probe_df = pd.read_csv(_probe_path)
_fe_probe = FeatureEngineer('temporal')
_probe_df_fe = _fe_probe.extract_features(_probe_df)

_feature_cols = [
    c for c in _probe_df_fe.columns
    if any(m in c.lower() for m in [
        'betweenness', 'pagerank', 'degree', 'closeness', 'eigenvector',
        'k_core', 'k_truss', 'modularity', 'flow_rate', 'avg_payload'
    ])
]
if not _feature_cols:
    _feature_cols = ['flow_rate', 'avg_payload_fwd', 'protocol_encoded']

INPUT_DIM  = len(_feature_cols)
NUM_CLASSES = df_sampled['Attack'].nunique()

print(f'input_dim   = {INPUT_DIM}')
print(f'num_classes = {NUM_CLASSES}')
print(f'features    : {_feature_cols}')

# ── Initialise system ─────────────────────────────────────────────────────────
fed_system = FedGATSageSystem(
    data_dir=DATA_OUTPUT_DIR,
    num_clients=NUM_CLIENTS,
    detector_types=DETECTOR_TYPES,
    device=DEVICE
)

fed_system.initialize_models(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES
)
print('Models initialised.')

# ── Run federated training ────────────────────────────────────────────────────
train_results = fed_system.train_federated(num_rounds=NUM_ROUNDS)

print('\nTraining complete.')
print(f'Final round loss : {train_results["training_losses"][-1]:.4f}')
print(f'Avg round time   : {np.mean(train_results["round_times"]):.1f}s')

## Evaluation & Results

Evaluate the global GraphSAGE model on the held-out test set from the `temporal` detector split. We compute:
- Overall accuracy, macro-F1, precision, recall
- Full classification report per attack class
- Confusion matrix heatmap
- Per-class F1 bar chart

In [ ]:
# ── Load test set ─────────────────────────────────────────────────────────────
test_csv = os.path.join(DATA_OUTPUT_DIR, 'temporal_detector', 'test.csv')
df_test = pd.read_csv(test_csv)
logger.info(f'Test set: {len(df_test):,} rows')

# Feature engineering + community features on test set
_fe_test = FeatureEngineer('temporal')
_ce_test = CentralityFeatureExtractor()
_cp_test = CommunityAwareProcessor()

df_test_fe = _fe_test.extract_features(df_test)
df_test_fe = _ce_test.extract_centrality_features(df_test_fe)
df_test_fe = _cp_test.create_community_enhanced_features(df_test_fe, {})

# Build label mapper from training data
_dl_eval = fed_system.data_loaders['temporal']
if _dl_eval.label_mapper is None:
    _dl_eval._create_label_mapper(df_test_fe)
label_mapper = _dl_eval.label_mapper
class_names = [k for k, _ in sorted(label_mapper.items(), key=lambda x: x[1])]

# Build graph tensors for test set
test_graph = _dl_eval._process_to_graph(df_test_fe)
print(f'Test graph: {test_graph["features"].shape[0]} nodes, '
      f'{test_graph["edge_index"].shape[1]} edges')

# ── Generate flow embeddings for test flows ───────────────────────────────────
# Use the temporal model from client 0 (representative model after FedAvg)
_temporal_model = fed_system.client_models['temporal'][0]
_flow_gen_eval = FlowEmbeddingGenerator('temporal')

test_embs, test_labels = _flow_gen_eval.generate_embeddings(
    _temporal_model, test_graph
)
print(f'Test flow embeddings: {test_embs.shape}')

# ── Predict with global GraphSAGE ─────────────────────────────────────────────
fed_system.global_model.eval()
with torch.no_grad():
    test_embs_dev = test_embs.to(DEVICE)
    test_edge_idx = fed_system._build_knn_edge_index(
        test_embs_dev, k=10, max_nodes=512
    )
    n_eval = test_edge_idx.max().item() + 1 if test_edge_idx.numel() > 0 \
             else test_embs_dev.shape[0]
    eval_x = test_embs_dev[:n_eval]
    eval_y = test_labels[:n_eval].to(DEVICE)
    _, logits = fed_system.global_model(eval_x, test_edge_idx)
    preds = logits.argmax(dim=1).cpu().numpy()
    true_labels = eval_y.cpu().numpy()

# ── Metrics ───────────────────────────────────────────────────────────────────
acc  = accuracy_score(true_labels, preds)
f1   = f1_score(true_labels, preds, average='macro', zero_division=0)
prec = precision_score(true_labels, preds, average='macro', zero_division=0)
rec  = recall_score(true_labels, preds, average='macro', zero_division=0)

print('\n=== Global Evaluation Metrics ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Macro-F1  : {f1:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')

# Determine which class names are actually present in eval split
present_classes = sorted(set(true_labels) | set(preds))
present_names   = [class_names[i] for i in present_classes if i < len(class_names)]

print('\n=== Per-class Report ===')
print(classification_report(
    true_labels, preds,
    labels=present_classes,
    target_names=present_names,
    zero_division=0
))

In [ ]:
# ── Training loss curve ───────────────────────────────────────────────────────
os.makedirs(RESULTS_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    range(1, len(train_results['training_losses']) + 1),
    train_results['training_losses'],
    marker='o', linewidth=2, color='steelblue'
)
axes[0].set_title('Global Model Loss per Round', fontsize=13)
axes[0].set_xlabel('Federated Round')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[0].grid(True, alpha=0.3)

axes[1].bar(
    range(1, len(train_results['round_times']) + 1),
    train_results['round_times'],
    color='salmon'
)
axes[1].set_title('Round Wall-Clock Time', fontsize=13)
axes[1].set_xlabel('Federated Round')
axes[1].set_ylabel('Time (s)')
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
loss_plot_path = os.path.join(RESULTS_DIR, 'training_loss.png')
plt.savefig(loss_plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {loss_plot_path}')

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(true_labels, preds, labels=present_classes)

fig, ax = plt.subplots(figsize=(max(6, len(present_names)),
                                max(5, len(present_names))))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=present_names, yticklabels=present_names,
    ax=ax, linewidths=0.5
)
ax.set_title('Confusion Matrix — FedGATSage (Global Model)', fontsize=13)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
cm_path = os.path.join(RESULTS_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {cm_path}')

# ── Per-class F1 bar chart ────────────────────────────────────────────────────
from sklearn.metrics import f1_score as _f1_score
per_class_f1 = _f1_score(
    true_labels, preds,
    labels=present_classes,
    average=None, zero_division=0
)

fig, ax = plt.subplots(figsize=(max(8, len(present_names) * 1.2), 5))
bars = ax.bar(present_names, per_class_f1, color='mediumseagreen', edgecolor='white')
ax.set_title('Per-Class F1 Score — FedGATSage', fontsize=13)
ax.set_xlabel('Attack Class')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.05)
ax.axhline(y=f1, color='red', linestyle='--', linewidth=1.5, label=f'Macro-F1 = {f1:.3f}')
ax.legend()
for bar, val in zip(bars, per_class_f1):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{val:.2f}', ha='center', va='bottom', fontsize=9
    )
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
f1_plot_path = os.path.join(RESULTS_DIR, 'per_class_f1.png')
plt.savefig(f1_plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {f1_plot_path}')

## Summary of Results

The cell below prints a clean summary and saves a JSON results file to `/kaggle/working/results/`.

In [ ]:
import json

summary = {
    'dataset': dataset_label,
    'num_clients': NUM_CLIENTS,
    'num_rounds': NUM_ROUNDS,
    'hidden_dim': HIDDEN_DIM,
    'num_classes': NUM_CLASSES,
    'input_dim': INPUT_DIM,
    'device': DEVICE,
    'evaluation': {
        'accuracy': round(float(acc), 4),
        'macro_f1': round(float(f1), 4),
        'macro_precision': round(float(prec), 4),
        'macro_recall': round(float(rec), 4),
        'per_class_f1': {
            name: round(float(score), 4)
            for name, score in zip(present_names, per_class_f1)
        }
    },
    'training': {
        'final_loss': round(float(train_results['training_losses'][-1]), 4),
        'min_loss': round(float(min(train_results['training_losses'])), 4),
        'total_training_time_s': round(float(sum(train_results['round_times'])), 1),
        'avg_round_time_s': round(float(np.mean(train_results['round_times'])), 1),
    }
}

summary_path = os.path.join(RESULTS_DIR, 'summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 55)
print('  FedGATSage Experiment Summary')
print('=' * 55)
print(f'  Dataset          : {summary["dataset"]}')
print(f'  Clients          : {summary["num_clients"]}')
print(f'  Rounds           : {summary["num_rounds"]}')
print(f'  Device           : {summary["device"]}')
print(f'  Classes          : {summary["num_classes"]}')
print('-' * 55)
print(f'  Accuracy         : {summary["evaluation"]["accuracy"]:.4f}')
print(f'  Macro-F1         : {summary["evaluation"]["macro_f1"]:.4f}')
print(f'  Macro-Precision  : {summary["evaluation"]["macro_precision"]:.4f}')
print(f'  Macro-Recall     : {summary["evaluation"]["macro_recall"]:.4f}')
print('-' * 55)
print(f'  Final round loss : {summary["training"]["final_loss"]:.4f}')
print(f'  Min round loss   : {summary["training"]["min_loss"]:.4f}')
print(f'  Total train time : {summary["training"]["total_training_time_s"]:.1f}s')
print(f'  Avg round time   : {summary["training"]["avg_round_time_s"]:.1f}s')
print('=' * 55)
print(f'\nResults saved to: {RESULTS_DIR}/')
print(f'  {loss_plot_path}')
print(f'  {cm_path}')
print(f'  {f1_plot_path}')
print(f'  {summary_path}')